In [ ]:
import pandas as pd

# adjust y-values based on driving direction
def adjust_y_values_dynamically(tracks_df, aligned_df):

    # Merge track and aligned data
    tracks_df = tracks_df.merge(aligned_df[['id', 'drivingDirection']], on='id', how='left')

    # Separate lanes based on direction
    right_to_left = tracks_df[tracks_df['drivingDirection'] == 1]['laneId'].unique()
    left_to_right = tracks_df[tracks_df['drivingDirection'] == 2]['laneId'].unique()

    # For right-to-left lanes, adjust x values and reverse velocities
    if 'drivingDirection' in tracks_df.columns:
        max_x = tracks_df['x'].max()
        tracks_df.loc[tracks_df['laneId'].isin(right_to_left), 'x'] = max_x - tracks_df['x']
        tracks_df.loc[tracks_df['laneId'].isin(right_to_left), ['xVelocity', 'xAcceleration']] *= -1

    # Adjust y-values for right-to-left lanes
    if not tracks_df[tracks_df['laneId'].isin(right_to_left)].empty:
        max_y = tracks_df[tracks_df['laneId'].isin(right_to_left)]['y'].max()
        tracks_df.loc[tracks_df['laneId'].isin(right_to_left), 'y'] = max_y - tracks_df.loc[tracks_df['laneId'].isin(right_to_left), 'y']

    # Adjust y-values for left-to-right lanes
    if not tracks_df[tracks_df['laneId'].isin(left_to_right)].empty:
        min_y = tracks_df[tracks_df['laneId'].isin(left_to_right)]['y'].min()
        tracks_df.loc[tracks_df['laneId'].isin(left_to_right), 'y'] -= min_y

    return tracks_df

# Function to process and adjust files
def process_files_without_lane_flipping(tracks_file, aligned_file, output_file):
    """
    Process and adjust files and save the output.
    """
    # Load input files
    tracks_df = pd.read_csv(tracks_file)
    aligned_df = pd.read_csv(aligned_file)

    # Adjust y values
    adjusted_tracks_df = adjust_y_values_dynamically(tracks_df, aligned_df)

    # Remove drivingDirection column
    adjusted_tracks_df = adjusted_tracks_df.drop(columns=['drivingDirection'], errors='ignore')

    # Save the output file
    adjusted_tracks_df.to_csv(output_file, index=False)
    print(f"File saved as '{output_file}'.")

# Main Execution
if __name__ == "__main__":
    # Input file paths
    tracks_file = "01_tracks.csv"
    aligned_file = "01_tracksMeta.csv"
    output_file = "dynamic_y_adjusted_tracks_new.csv"

    # Run the process
    process_files_without_lane_flipping(tracks_file, aligned_file, output_file)



File saved as 'dynamic_y_adjusted_tracks_new.csv'.
